# Notatki
Problem z FactMentalHealt jest taki że mamy dane co roku więc nie mamy jak analizować związku z pogodą, 
możemy dodać sztuczne dane co miesiąc ale wtedy trzeba pomyśleć jak bo to będzie ważne i wpłynie na wnioski mocno  
Zakładam że FactWeather zajmie się Kawa xd w sensie ja nie wiem jak to działa więc tego nie ruszam na razie  
DimDate dodam potem skryptem z teams ale to jest skrypt SQL więc to potem  

W planie bazy danych nie mam np w FactMentalHealt nazw zaburzeń tylko key (int) ale no to też zrobimy w SQL bo bez sensu teraz

In [1]:
# pip install pycountry

In [2]:
import pandas as pd
import numpy as np

import pycountry

In [3]:
df_al = pd.read_csv("original_data/alcohol_consumption_dataset.csv")
df_un = pd.read_csv("original_data/unemployment_dataset.csv")
df_mh = pd.read_csv("original_data/mental_health_dataset.csv")

In [4]:
def fill_missing_values(df, missing_countries, neighbors, target):
    new_rows = []
    for country in missing_countries:
        for category_val in df[target].unique():
            for year in df['year'].unique():
                # znajdź wartości sąsiadów
                mask = (
                    df['country'].isin(neighbors[country]) &
                    (df[target] == category_val) &
                    (df['year'] == year)
                )
                neighbor_vals = df[mask]['value'].values
                
                if len(neighbor_vals) > 0:
                    val = neighbor_vals.mean()
                    new_rows.append({
                        'country': country,
                        'year': year,
                        target: category_val,
                        'value': val
                    })
    return pd.DataFrame(new_rows)

In [5]:
df_al.head(2)

,Entity,Code,Year,"Total alcohol consumption per capita (liters of pure alcohol, projected estimates, 15+ years of age)","GDP per capita, PPP (constant 2017 international $)",Population (historical estimates),Continent
0,Abkhazia,OWID_ABK,2015,NaN,NaN,NaN,Asia
1,Afghanistan,AFG,2010,0.21,1957.02907,29185511.0,NaN


In [6]:
df_un.head(2)

,Country Name,Country Code,1991,1992,1993,1994,1995,1996,1997,1998,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
0,Africa Eastern and Southern,AFE,7.80,7.84,7.85,7.84,7.83,7.84,7.86,7.81,...,6.56,6.45,6.41,6.49,6.61,6.71,6.73,6.91,7.56,8.11
1,Afghanistan,AFG,10.65,10.82,10.72,10.73,11.18,10.96,10.78,10.80,...,11.34,11.19,11.14,11.13,11.16,11.18,11.15,11.22,11.71,13.28


In [7]:
df_mh.head(2)

,Entity,Code,Year,Schizophrenia (%),Bipolar disorder (%),Eating disorders (%),Anxiety disorders (%),Drug use disorders (%),Depression (%),Alcohol use disorders (%)
0,Afghanistan,AFG,1990,"0,1605595422","0,6977793845","0,1018548635","4,828829705","1,677081945","4,071831177","0,6724040862"
1,Afghanistan,AFG,1991,"0,1603118986","0,6979605949","0,0993127901","4,829740372","1,684745731","4,079530935","0,6717681239"


# DimCountry

https://en.wikipedia.org/wiki/List_of_European_countries_by_average_wage  
https://www.barenbrug.com/climate-zones-europe-and-mediterranean

In [8]:
# Gross average monthly earnings - climate zones

data = [
    ("AL", "Albania", "Low", "Mediterranean climate"),
    ("AD", "Andorra", "Middle", "Mediterranean climate"),
    ("AT", "Austria", "Higher middle", "Eastern-continental climate"),
    ("BY", "Belarus", "Low", "Eastern-continental climate"),
    ("BE", "Belgium", "Higher middle", "Oceanic climate"),
    ("BA", "Bosnia and Herzegovina", "Lower middle", "Mediterranean climate"),
    ("BG", "Bulgaria", "Lower middle", "Mediterranean climate"),
    ("HR", "Croatia", "Middle", "Mediterranean climate"),
    ("CY", "Cyprus", "Middle", "Mediterranean climate"),
    ("CZ", "Czech Republic", "Eastern-continental climate"),
    ("DK", "Denmark", "High", "Oceanic climate"),
    ("EE", "Estonia", "Middle", "Nordic climate"),
    ("FI", "Finland", "Higher middle", "Nordic climate"),
    ("FR", "France", "Higher middle", "Oceanic climate"),
    ("DE", "Germany", "Higher middle", "Oceanic climate"),
    ("GR", "Greece", "Lower middle", "Mediterranean climate"),
    ("HU", "Hungary", "Lower middle", "Eastern-continental climate"),
    ("IS", "Iceland", "High", "Nordic climate"),
    ("IE", "Ireland", "High", "Oceanic climate"),
    ("IT", "Italy", "Middle", "Mediterranean climate"),
    ("LV", "Latvia", "Lower middle", "Eastern-continental climate"),
    ("LI", "Liechtenstein", "High", "Eastern-continental climate"),
    ("LT", "Lithuania", "Higher middle", "Eastern-continental climate"),
    ("LU", "Luxembourg", "High", "Oceanic climate"),
    ("MT", "Malta", "Higer middle", "Mediterranean climate"),
    ("MD", "Moldova", "Low", "Mediterranean climate"),
    ("MC", "Monaco", "High", "Mediterranean climate"),
    ("ME", "Montenegro", "Lower middle", "Mediterranean climate"),
    ("NL", "Netherlands", "Higher middle", "Oceanic climate"),
    ("MK", "North Macedonia", "Lower middle", "Mediterranean climate"),
    ("NO", "Norway", "Higher middle", "Nordic climate"),
    ("PL", "Poland", "Middle", "Eastern-continental climate"),
    ("PT", "Portugal", "Middle", "Mediterranean climate"),
    ("RO", "Romania", "Lower middle", "Eastern-continental climate"),
    ("RU", "Russia", "Lower middle", "Eastern-continental climate"),
    ("SM", "San Marino", "Middle", "Mediterranean climate"),
    ("RS", "Serbia", "Lower middle", "Eastern-continental climate"),
    ("SK", "Slovakia", "Higher middle", "Eastern-continental climate"),
    ("SI", "Slovenia", "Middle", "Mediterranean climate"),
    ("ES", "Spain", "Middle", "Mediterranean climate"),
    ("SE", "Sweden", "Higher middle", "Nordic climate"),
    ("CH", "Switzerland", "High", "Eastern-continental climate"),
    ("UA", "Ukraine", "Low", "Eastern-continental climate"),
    ("GB", "United Kingdom", "Higher middle", "Oceanic climate"),
    ("VA", "Vatican City", "Higher middle", "Mediterranean climate"),
]

DimCountry = pd.DataFrame(data, columns=["CountryCode", "CountryName", "IncomeGroup", "ClimateType"])
DimCountry["CountryKey"] = range(1, len(DimCountry)+1)

# Przesuń CountryKey na początek
DimCountry = DimCountry[["CountryKey", "CountryCode", "CountryName", "IncomeGroup", "ClimateType"]]

In [9]:
DimCountry.to_csv("final_data/DimCountry.csv", index=False)

# DimDisorder

In [10]:
data = [
    # Schizophrenia
    ("Schizophrenia", "Schizophrenia", "Antipsychotics", "Psychotherapy"),
    
    # Bipolar
    ("Bipolar disorder", "Bipolar disorder", "Mood stabilizers", "Psychotherapy"),
    
    # Eating disorders
    ("Anorexia", "Eating disorders", "None", "Nutritional therapy"),
    ("Bulimia", "Eating disorders", "SSRIs", "CBT"), # terapia poznawczo behawioralna
    ("Binge eating disorder", "Eating disorders", "SSRIs", "CBT"),
    
    # Anxiety
    ("Generalized anxiety disorder", "Anxiety disorders", "SSRIs", "CBT"),
    ("Panic disorder", "Anxiety disorders", "SSRIs", "CBT"),
    ("Social anxiety disorder", "Anxiety disorders", "SSRIs", "CBT"),
    
    # Drug use
    ("Opioid use disorder", "Drug use disorders", "Methadone", "Behavioral therapy"),
    ("Cannabis use disorder", "Drug use disorders", "None", "Behavioral therapy"),
    ("Stimulant use disorder", "Drug use disorders", "None", "Behavioral therapy"),
    
    # Depression
    ("Major depressive disorder", "Depression", "SSRIs", "CBT"),
    ("Persistent depressive disorder", "Depression", "SSRIs", "Psychotherapy"),
    
    # Alcohol
    ("Alcohol dependence", "Alcohol use disorders", "Disulfiram", "Behavioral therapy"),
    ("Alcohol abuse", "Alcohol use disorders", "None", "Behavioral therapy"),
]

DimDisorder = pd.DataFrame(
    data,
    columns=["DisorderName", "Category", "Meds", "Therapy"]
)

DimDisorder["DisorderKey"] = range(1, len(DimDisorder)+1)

DimDisorder = DimDisorder[
    ["DisorderKey", "DisorderName", "Category", "Meds", "Therapy"]
]


In [11]:
DimDisorder.to_csv("final_data/DimDisorder.csv", index=False)

# DimMonth

In [12]:
rows = []

for year in range(2000, 2011):  # 2000–2010
    for month in range(1, 13):
        month_key = year * 100 + month
        quarter = (month - 1) // 3 + 1
        
        rows.append((month_key, year, month, quarter))

DimMonth = pd.DataFrame(
    rows,
    columns=["MonthKey", "Year", "Month", "Quarter"]
)

In [13]:
DimMonth.to_csv("final_data/DimMonth.csv", index=False)

# FactMentalHealth
Tutaj musimy wybrać tylko kraje Europy, zobaczyć czy mamy dla nich pełne dane i rozszerzyć o dokładniejsze kategorie zaburzeń poprzez podział danych w istniejących kategoriach

In [14]:
# Wybieramy tylko kraje europejskie
europe_countries = set(DimCountry["CountryName"])

df_mh = df_mh[df_mh["Entity"].isin(europe_countries)]

In [15]:
# Wybieram tylko lata 2000-2010
df_mh = df_mh[
    (df_mh["Year"] >= 2000) &
    (df_mh["Year"] <= 2010)
]

In [16]:
# Zmiana na long format
df_mh_long = df_mh.melt(
    id_vars=["Entity", "Code", "Year"],
    var_name="disorder",
    value_name="value"
)

In [17]:
# Usuwanie % z nazw
df_mh_long["disorder"] = df_mh_long["disorder"].str.replace(" (%)", "", regex=False)

In [18]:
# Ujednolicenie nazw
df_mh_long = df_mh_long.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year"
})

In [19]:
# Ujednolicenie nazw
name_map = {
    "Eating disorders": "Eating disorders",
    "Anxiety disorders": "Anxiety",
    "Drug use disorders": "Drug use",
    "Alcohol use disorders": "Alcohol use",
}

df_mh_long["disorder"] = df_mh_long["disorder"].replace(name_map)

In [20]:
df_mh_long

,country,code,year,disorder,value
0,Albania,ALB,2000,Schizophrenia,"0,1945407797"
1,Albania,ALB,2001,Schizophrenia,"0,1948120831"
2,Albania,ALB,2002,Schizophrenia,"0,1952144122"
3,Albania,ALB,2003,Schizophrenia,"0,1956773182"
4,Albania,ALB,2004,Schizophrenia,"0,1961393927"
...,...,...,...,...,...
3075,United Kingdom,GBR,2006,Alcohol use,"1,761617083"
3076,United Kingdom,GBR,2007,Alcohol use,"1,806324297"
3077,United Kingdom,GBR,2008,Alcohol use,"1,852339401"
3078,United Kingdom,GBR,2009,Alcohol use,"1,889700918"


In [21]:
# jeśli są przecinki w danych
df_mh_long["value"] = df_mh_long["value"].str.replace(",", ".").astype(float)

In [22]:
# Podział procentów na podkategorie
split_config = {
    "Eating disorders": {
        "Anorexia": 0.35,
        "Bulimia": 0.35,
        "Binge eating disorder": 0.30,
    },
    "Anxiety": {
        "Generalized anxiety disorder": 0.4,
        "Panic disorder": 0.3,
        "Social anxiety disorder": 0.3,
    },
    "Drug use": {
        "Opioid use disorder": 0.4,
        "Cannabis use disorder": 0.35,
        "Stimulant use disorder": 0.25,
    },
    "Depression": {
        "Major depressive disorder": 0.75,
        "Persistent depressive disorder": 0.25,
    },
    "Alcohol use": {
        "Alcohol dependence": 0.6,
        "Alcohol abuse": 0.4,
    }
}


In [23]:
# Do aplikowania podziału
def split_value(value, mapping):
    keys = list(mapping.keys())
    base = np.array(list(mapping.values()))
    
    # dodaj lekki szum
    noise = np.random.normal(1, 0.05, len(base))
    proportions = base * noise
    
    # normalizacja żeby suma = 1
    proportions = proportions / proportions.sum()
    
    return dict(zip(keys, value * proportions))

In [24]:
# Uzupełnienie podkategorii
new_rows = []

for _, row in df_mh_long.iterrows():
    disorder = row["disorder"]
    value = row["value"]
    year = row["year"]   # <-- dodajemy tutaj
    
    if disorder in split_config:
        split_vals = split_value(value, split_config[disorder])
        
        for sub_disorder, val in split_vals.items():
            new_rows.append({
                "country": row["country"],
                "year": year,           # <-- kopiujemy year
                "disorder": sub_disorder,
                "value": val
            })
    else:
        # Schizophrenia i Bipolar zostają bez zmian
        new_rows.append({
            "country": row["country"],
            "year": year,           # <-- kopiujemy year
            "disorder": disorder,
            "value": value
        })

df_mh_expanded = pd.DataFrame(new_rows)

In [25]:
df_mh_expanded

,country,year,disorder,value
0,Albania,2000,Schizophrenia,0.194541
1,Albania,2001,Schizophrenia,0.194812
2,Albania,2002,Schizophrenia,0.195214
3,Albania,2003,Schizophrenia,0.195677
4,Albania,2004,Schizophrenia,0.196139
...,...,...,...,...
6595,United Kingdom,2008,Alcohol abuse,0.742710
6596,United Kingdom,2009,Alcohol dependence,1.144760
6597,United Kingdom,2009,Alcohol abuse,0.744941
6598,United Kingdom,2010,Alcohol dependence,1.110220


In [26]:
# Znalezienie braków
missing_countries = europe_countries - set(df_mh["Entity"].unique())
print(missing_countries)

{'Vatican City', 'Liechtenstein', 'North Macedonia', 'Monaco', 'San Marino'}


In [27]:
# Uzupełnienie braków
missing_countries = ['Vatican City', 'San Marino', 'Liechtenstein', 'Monaco', 'North Macedonia']

neighbors = {
    "Vatican City": ["Italy"],
    "San Marino": ["Italy"],
    "Liechtenstein": ["Switzerland", "Austria"],
    "Monaco": ["France"],
    "North Macedonia": ["Greece", "Bulgaria"]
}

In [28]:
df_missing = fill_missing_values(df_mh_expanded, missing_countries, neighbors, target="disorder")
df_mh_expanded = pd.concat([df_mh_expanded, df_missing], ignore_index=True)

In [29]:
df_mh_expanded

,country,year,disorder,value
0,Albania,2000,Schizophrenia,0.194541
1,Albania,2001,Schizophrenia,0.194812
2,Albania,2002,Schizophrenia,0.195214
3,Albania,2003,Schizophrenia,0.195677
4,Albania,2004,Schizophrenia,0.196139
...,...,...,...,...
7420,North Macedonia,2006,Alcohol abuse,0.539523
7421,North Macedonia,2007,Alcohol abuse,0.602491
7422,North Macedonia,2008,Alcohol abuse,0.575283
7423,North Macedonia,2009,Alcohol abuse,0.630102


In [30]:
df_mh_expanded.to_csv("final_data/FactMentalHealthYearly.csv", index=False)

# FactEconomy

In [31]:
# Wybieramy tylko kraje europejskie
df_al = df_al[df_al["Entity"].isin(europe_countries)]
df_un = df_un[df_un["Country Name"].isin(europe_countries)]

In [32]:
# ALKOHOL
# Wybieram tylko lata 2000-2010
df_al = df_al[
    (df_al["Year"] >= 2000) &
    (df_al["Year"] <= 2010)
]

df_al.head(3)

,Entity,Code,Year,"Total alcohol consumption per capita (liters of pure alcohol, projected estimates, 15+ years of age)","GDP per capita, PPP (constant 2017 international $)",Population (historical estimates),Continent
582,Albania,ALB,2000,6.57,5893.136233,3129246.0,NaN
583,Albania,ALB,2005,7.65,8040.878717,3086810.0,NaN
584,Albania,ALB,2010,7.69,10749.487448,2948029.0,NaN


In [33]:
df_al = df_al.drop(columns=["Continent"])

In [34]:
df_al = df_al.rename(columns={
    "Entity": "country",
    "Code": "code",
    "Year": "year",
    "Total alcohol consumption per capita (liters of pure alcohol, projected estimates, 15+ years of age)": "alkohol_consumption",
    "GDP per capita, PPP (constant 2017 international $)": "GDP",
    "Population (historical estimates)": "population",
})

In [35]:
# Znalezienie braków w wartościach
df_al.groupby('country')[['alkohol_consumption', 'GDP', 'population']].apply(lambda x: x.isna().sum())

,alkohol_consumption,GDP,population
country,,,
Albania,8,0,0
Andorra,8,11,0
Austria,8,0,0
Belarus,8,0,0
Belgium,8,0,0
Bosnia and Herzegovina,8,0,0
Bulgaria,8,0,0
Croatia,8,0,0
Cyprus,8,0,0


In [36]:
# Interpolacja alkoholu dla wszystkich krajów 
df_al_interp = df_al.sort_values(['country', 'year']).set_index(['country', 'year'])

df_al_interp['alkohol_consumption'] = df_al_interp.groupby('country')['alkohol_consumption'].transform(lambda x: x.interpolate(method='linear'))

df_al = df_al_interp.reset_index()

In [37]:
# Zastąpienie brakujących wartości alkohol średnią z sąsiadów

countries_neighbors = {
    "Serbia": ['Croatia', 'Bosnia and Herzegovina', 'Hungary'],
    "Liechtenstein": ["Switzerland", "Austria"],
    "Monaco": ["France"],
    "Montenegro": ["Croatia", "Bosnia and Herzegovina", "Albania"],
    "San Marino": ["Italy"]
}

for country, neighbors in countries_neighbors.items():
    for year in range(2000, 2011):
        
        # średnia z sąsiadów
        val = df_al.loc[
            (df_al['country'].isin(neighbors)) & (df_al['year'] == year),
            'alkohol_consumption'
        ].mean()
        
        # maska dla kraju
        mask = (df_al['country'] == country) & (df_al['year'] == year)
        
        # uzupełniamy tylko brakujące wartości
        df_al.loc[mask, 'alkohol_consumption'] = val

In [38]:
# Zastąpienie brakujących części GDP podbitą średnią z sąsiadów
countries_neighbors = {
    "Monaco": ["France"],
    "Liechtenstein": ["Switzerland", "Austria"],
    "Andorra": ["Spain", "France"]
}

for country, neighbors in countries_neighbors.items():
    for year in range(2000, 2011):
        
        # średnia GDP z sąsiadów
        val = df_al.loc[
            (df_al['country'].isin(neighbors)) & (df_al['year'] == year),
            'GDP'
        ].mean()
        
        mask = (df_al['country'] == country) & (df_al['year'] == year)

        val = val*1.2 # Bo te małe są bogatsze 
        
        # uzupełniamy tylko brakujące wartości
        df_al.loc[mask & df_al['GDP'].isna(), 'GDP'] = val

In [39]:
df_al.isna().sum()

country                0
year                   0
code                   0
alkohol_consumption    0
GDP                    0
population             0
dtype: int64

In [40]:
# Znalezienie braków w krajach
missing_countries = europe_countries - set(df_al["country"].unique())
print("Alkohol: ", missing_countries)

Alkohol:  {'Vatican City', 'Czech Republic'}


In [41]:
for year in range(2000, 2011):
    # Vatican City
    df_al = pd.concat([
        df_al,
        pd.DataFrame([{
            'country': 'Vatican City',
            'code': 'VA',
            'year': year,
            'alkohol_consumption': df_al.loc[(df_al['country']=='Italy') & (df_al['year']==year), 'alkohol_consumption'].values[0],
            'GDP': df_al.loc[(df_al['country']=='Italy') & (df_al['year']==year), 'GDP'].values[0],
            'population': 900
        }])
    ], ignore_index=True)
    
    # Czech Republic
    df_al = pd.concat([
        df_al,
        pd.DataFrame([{
            'country': 'Czech Republic',
            'code': 'CZ',
            'year': year,
            'alkohol_consumption': df_al.loc[(df_al['country']=='Poland') & (df_al['year']==year), 'alkohol_consumption'].values[0] * 1.1,
            'GDP': df_al.loc[(df_al['country']=='Poland') & (df_al['year']==year), 'GDP'].values[0] * 1.1,
            'population': df_al.loc[(df_al['country']=='Poland') & (df_al['year']==year), 'population'].values[0] * 0.3
        }])
    ], ignore_index=True)

In [42]:
df_al[df_al['country']=='Czech Republic'].head(3)

,country,year,code,alkohol_consumption,GDP,population
474,Czech Republic,2000,CZ,10.0870,17883.410464,11567009.7
476,Czech Republic,2001,CZ,10.4016,18113.464596,11558875.5
478,Czech Republic,2002,CZ,10.7162,18490.800015,11546592.0


In [43]:
df_al.head(3)

,country,year,code,alkohol_consumption,GDP,population
0,Albania,2000,ALB,6.570,5893.136233,3129246.0
1,Albania,2001,ALB,6.786,6441.853452,3129701.0
2,Albania,2002,ALB,7.002,6754.536003,3126183.0


In [44]:
# UNEMPLOYMENT
df_un_long = df_un.melt(
    id_vars=["Country Name", "Country Code"],
    value_vars=[str(y) for y in range(2000, 2011)],
    var_name="Year",
    value_name="Unemployment"
)
df_un_long.head(3)

,Country Name,Country Code,Year,Unemployment
0,Albania,ALB,2000,19.03
1,Austria,AUT,2000,4.69
2,Belgium,BEL,2000,6.59


In [45]:
df_un_long = df_un_long.drop(columns=["Country Code"])

In [46]:
df_un_long = df_un_long.rename(columns={
    "Country Name": "country",
    "Year": "year",
    "Unemployment": "unemployment",
})

In [47]:
df_un_long.head(2)

,country,year,unemployment
0,Albania,2000,19.03
1,Austria,2000,4.69


In [48]:
df_un_long.isna().sum()

country         0
year            0
unemployment    0
dtype: int64

In [49]:
# Musimy tylko dodać brakujące kraje
missing_countries = europe_countries - set(df_un_long["country"].unique())
print("Unemployment: ", missing_countries)

Unemployment:  {'Vatican City', 'Liechtenstein', 'Russia', 'Monaco', 'San Marino', 'Slovakia', 'Andorra'}


In [50]:
df_un_long.head(1)

,country,year,unemployment
0,Albania,2000,19.03


In [51]:
df_un_long.dtypes

country          object
year             object
unemployment    float64
dtype: object

In [52]:
# Konwertujemy year na int
df_un_long['year'] = df_un_long['year'].astype(int)

# Opcjonalnie usuń spacje w country
df_un_long['country'] = df_un_long['country'].str.strip()

In [53]:
countries_neighbors = {
    "Vatican City": ["Italy"],
    "San Marino": ["Italy"],
    "Monaco": ["France"],
    "Liechtenstein": ["Switzerland", "Austria"],
    "Andorra": ["Spain", "France"],
    "Slovakia": ["Czech Republic", "Poland"],
    "Russia": ["Poland", "Hungary"]
}

new_rows = []

for country, neighbors in countries_neighbors.items():
    for year in range(2000, 2011):
        
        val = df_un_long.loc[
            (df_un_long['country'].isin(neighbors)) & (df_un_long['year'] == year),
            'unemployment'
        ].mean()
        
        mask = (df_un_long['country'] == country) & (df_un_long['year'] == year)
        
        if mask.sum() == 0:
            new_rows.append({
                'country': country,
                'year': year,
                'unemployment': val
            })
        else:
            df_un_long.loc[mask & df_un_long['unemployment'].isna(), 'unemployment'] = val

# dodanie wszystkich nowych wierszy
df_un_long = pd.concat([df_un_long, pd.DataFrame(new_rows)], ignore_index=True)

In [54]:
# Musimy tylko dodać brakujące kraje
missing_countries = europe_countries - set(df_un_long["country"].unique())
print("Unemployment: ", missing_countries)

Unemployment:  set()


In [55]:
df_un_long.isna().sum()

country         0
year            0
unemployment    0
dtype: int64

In [56]:
df_economy = pd.merge(
    df_al,
    df_un_long,
    on=['country', 'year'],
    how='inner'  # tylko wspólne rekordy
)

In [57]:
df_economy.head()

,country,year,code,alkohol_consumption,GDP,population,unemployment
0,Albania,2000,ALB,6.570,5893.136233,3129246.0,19.03
1,Albania,2001,ALB,6.786,6441.853452,3129701.0,18.58
2,Albania,2002,ALB,7.002,6754.536003,3126183.0,17.90
3,Albania,2003,ALB,7.218,7154.784825,3118017.0,16.99
4,Albania,2004,ALB,7.434,7580.629091,3104893.0,16.31


In [59]:
# parametry w zależności od GDP
def generate_health_exp(row):
    gdp = row['GDP']
    if gdp < 20000:  # biedniejsze
        return gdp * np.random.uniform(0.05, 0.07)
    elif gdp < 40000:  # średnia
        return gdp * np.random.uniform(0.07, 0.09)
    else:  # bogatsze
        return gdp * np.random.uniform(0.09, 0.12)

df_economy['HealthExpenditure'] = df_economy.apply(generate_health_exp, axis=1)

In [60]:
df_economy.head()

,country,year,code,alkohol_consumption,GDP,population,unemployment,HealthExpenditure
0,Albania,2000,ALB,6.570,5893.136233,3129246.0,19.03,355.151333
1,Albania,2001,ALB,6.786,6441.853452,3129701.0,18.58,361.677533
2,Albania,2002,ALB,7.002,6754.536003,3126183.0,17.90,366.629136
3,Albania,2003,ALB,7.218,7154.784825,3118017.0,16.99,487.504488
4,Albania,2004,ALB,7.434,7580.629091,3104893.0,16.31,467.545759


In [62]:
df_economy.to_csv("final_data/FactEconomyYearly.csv", index=False)